# Component 1 — GIK Catalog & IceChunk Virtual Store

C1 ingests GIK Parquet key-reference files produced by ECMWF IFS ensemble output
and commits them as virtual Zarr arrays into an IceChunk store with full time-travel support.

**Key classes**: `GIKCatalog`, `IceChainStore`

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import date

from gik_icechain.shared.config import load_config

# Load .env credentials
_env = Path("../.env")
if _env.exists():
    for _line in _env.read_text().splitlines():
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("AWS_ACCESS_KEY_ID", os.environ.get("MINIO_ACCESS_KEY", ""))
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", os.environ.get("MINIO_SECRET_KEY", ""))

cfg = load_config(Path("../configs/default.yaml"))
STORAGE_OPTIONS = {"endpoint_url": cfg.outputs.endpoint_url}

# Demo window — data available in the store
START = date(2025, 1, 1)
END   = date(2025, 1, 2)
print(f"Config loaded — endpoint: {cfg.outputs.endpoint_url}")
print(f"Demo window: {START} → {END}")

## 1.1 GIK Catalog

In [ ]:
from gik_icechain.conversion.gik_loader import GIKCatalog

catalog = GIKCatalog(
    cfg.sources.gik_hf_dataset,
    catalog_file=cfg.sources.gik_catalog_file,
)
cat_df = catalog.load_catalog()
print(f"Catalog rows  : {len(cat_df)}")
print(f"Date range    : {cat_df['date'].min()} → {cat_df['date'].max()}")
print(cat_df.head(3))

paths = catalog.get_parquet_paths(start=START, end=END, run_hours=(0,), variables=["tp"])
print(f"\nParquet paths for {START} run-00Z tp: {len(paths)}")

## 1.2 IceChunk store — snapshots

In [ ]:
from gik_icechain.conversion.icechunk_writer import IceChainStore

store = IceChainStore(
    cfg.outputs.icechunk_store_uri,
    region=cfg.outputs.icechunk_store_region,
    endpoint_url=cfg.outputs.endpoint_url,
)
store.create_or_open()

snapshots = store.list_snapshots()
snap_df = pd.DataFrame(snapshots)
print(snap_df.to_string(index=False))
print(f"\nTotal snapshots: {len(snap_df)}")

## 1.3 Checkout a forecast day

In [ ]:
ds = store.checkout_as_of(START)
print(ds)
print(f"\ntp shape: {ds['tp'].shape}")
print(f"Members: {ds.sizes['member']}, Steps: {ds.sizes['step']}")

## 1.4 Store validation

In [ ]:
report = store.validate()
print(report)
assert report["committed_days"] >= 7, f"Expected >=7 committed days, got {report['committed_days']}"
assert report["gaps_detected"] == 0, f"Expected 0 gaps, got {report['gaps_detected']}: {report['gap_details']}"
print("Validation passed")

## 1.5 Time-travel comparison

In [ ]:
ds_jan1 = store.checkout_as_of(date(2025, 1, 1))
ds_jan7 = store.checkout_as_of(date(2025, 1, 7))
print(f"Jan 1 tp mean: {float(ds_jan1['tp'].mean()):.5f} m")
print(f"Jan 7 tp mean: {float(ds_jan7['tp'].mean()):.5f} m")

## 1.6 Visualize ensemble spread (step=4, first member vs last member)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ds_jan1["tp"].isel(step=4, member=0).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Member 0")
ds_jan1["tp"].isel(step=4, member=-1).plot(ax=axes[1], cmap="Blues")
axes[1].set_title("Member 49")
fig.suptitle("TP at step 24h — 2025-01-01", fontweight="bold")
fig.tight_layout()
plt.show()